# Analyse complète — Avis Abritel (3 sources)

**Périmètre des données**

- Sources : Google Play, App Store (RSS), Trustpilot
- Période : depuis le 01/01/2025 (scraping incrémental)
- Catégorisation : mots-clés + validation IA (Ollama gemma4:31b)
- Gravité : Haute / Moyenne / Basse (calculée depuis note + catégorie)
- Version release : rattachée à chaque avis via `data/releases.csv`

**Méthodologie de catégorisation**

Le pipeline applique deux étapes de catégorisation :
1. **Mots-clés** (`Catégorie_mots_cles`) : règles lexicales sur le texte normalisé
2. **IA — Ollama** (`Catégorie_ollama`) : validation / correction par LLM (gemma4:31b)

La colonne `Catégorie` (finale) correspond à la catégorie Ollama. Ce notebook compare les deux approches et analyse l'apport de l'IA.

In [ ]:
# Configuration : chemins, style des graphiques, affichage des tableaux
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

# Paramètres d'affichage
plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "figure.dpi": 120,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
    }
)
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}".rstrip("0").rstrip("."))

DATA_PATH = Path("data") / "avis_enrichis.csv"
RELEASES_PATH = Path("data") / "releases.csv"
RANDOM_STATE = 42
ALPHA = 0.05

COLONNES_ATTENDUES = [
    "date",
    "note",
    "texte",
    "source",
    "longueur_texte",
    "n_caracteres",
    "Catégorie",
    "Catégorie_secondaire",
    "Catégorie_mots_cles",
    "Catégorie_ollama",
    "Gravité",
    "Gravité_texte",
    "Autre_type",
    "version_release",
]
ORDRE_GRAVITE = ["Haute", "Moyenne", "Basse"]
ORDRE_NOTES = [1, 2, 3, 4, 5]
COULEURS_SRC = {"Google Play": "steelblue", "App Store": "coral", "Trustpilot": "mediumpurple"}

## 1. Chargement et schéma

On charge le CSV en **UTF-8-SIG** et on parse la colonne `date`. Si le fichier est absent,
relancer le pipeline : `uv run python 1_pipeline.py`

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Fichier introuvable : {DATA_PATH.resolve()}\n"
        "Lance le pipeline depuis la racine du projet : uv run python 1_pipeline.py"
    )

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

manquantes = [c for c in COLONNES_ATTENDUES if c not in df.columns]
if manquantes:
    raise ValueError(f"Colonnes manquantes : {manquantes}. Colonnes lues : {list(df.columns)}")

# Typage explicite
df["note"] = pd.to_numeric(df["note"], errors="coerce").astype("Int64")
df["texte"] = df["texte"].astype(str)
df["Catégorie_secondaire"] = df["Catégorie_secondaire"].fillna("").astype(str)
df["Autre_type"] = df["Autre_type"].fillna("").astype(str)

N = len(df)
print("Dimensions :", N, "lignes ×", df.shape[1], "colonnes")
print("Période couverte :", df["date"].min(), "→", df["date"].max())
df.dtypes.to_frame("dtype")

In [ ]:
df.head(8)

## 2. Contrôle qualité

Vérifications pour des indicateurs fiables : **doublons** (même définition que le pipeline),
**valeurs manquantes**, **anomalies structurelles** (notes hors [1,5], catégories inconnues),
et **cohérence de la catégorisation** (Catégorie ≡ Catégorie_ollama).

In [ ]:
from abritel.pipeline import valider_dataframe

# Doublons pipeline (source+date+note+texte)
dup_pipeline = df.duplicated(subset=["source", "date", "note", "texte"], keep=False)
n_dup = int(dup_pipeline.sum())
print(f"Doublons pipeline (source+date+note+texte) : {n_dup}")

# Validation structurelle
anomalies = valider_dataframe(df)
if anomalies:
    for a in anomalies:
        print(f"  ⚠ {a}")
else:
    print("✓ Aucune anomalie structurelle détectée")

# Valeurs manquantes
missing = df.isnull().sum()
missing_pct = (100 * missing / N).round(1)
miss_df = pd.DataFrame({"manquants": missing, "%": missing_pct})
cols_with_miss = miss_df[miss_df["manquants"] > 0]
if not cols_with_miss.empty:
    print("\nValeurs manquantes :")
    display(cols_with_miss)
else:
    print("\n✓ Aucune valeur manquante (hors champs optionnels)")

# Cohérence Catégorie ≡ Catégorie_ollama
accord_final = (df["Catégorie"] == df["Catégorie_ollama"]).all()
print(
    f"\nCohérence Catégorie ≡ Catégorie_ollama : {'✓ 100%' if accord_final else '✗ Incohérence !'}"
)

## 3. Biais de plateforme — diagnostic critique

Les trois sources **n'ont pas le même biais de sélection** :

| Source | Mécanisme | Biais attendu |
|--------|-----------|---------------|
| **Google Play** | Pagination chronologique, toutes notes | Distribution équilibrée |
| **App Store** | RSS Apple — 500 avis les plus récents | Sur-représentation des 1★ récents |
| **Trustpilot** | Pagination par étoiles 1–5 | Très forte sur-représentation 1★ — plateforme de signalement |

**Conséquence** : toute statistique agrégée dépend du poids relatif de chaque source.
→ **Toujours lire les métriques par source. L'agrégation brute est trompeuse.**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, src in zip(axes, ["Google Play", "App Store", "Trustpilot"], strict=False):
    sub = df[df["source"] == src]
    counts = sub["note"].value_counts().reindex(ORDRE_NOTES, fill_value=0)
    pct = 100 * counts / counts.sum()
    bars = ax.bar(counts.index.astype(str), pct.values, color=COULEURS_SRC[src], edgecolor="white")
    ax.set_title(
        f"{src}\n(n={len(sub)}, moy={sub['note'].mean():.2f}, méd={sub['note'].median():.0f}★)"
    )
    ax.set_ylabel("% des avis")
    ax.set_xlabel("Note")
    for bar, p in zip(bars, pct.values, strict=False):
        if p > 3:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 1,
                f"{p:.0f}%",
                ha="center",
                va="bottom",
                fontsize=9,
            )

fig.suptitle("Distribution des notes par source — Diagnostic de biais", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Indicateurs clés (KPIs) — à reproduire dans Power BI

**Définitions** (dénominateur = nombre d'avis de la population considérée) :
- **Note moyenne / médiane** : mesure centrale de satisfaction
- **Promoteurs** : note ∈ {4, 5}
- **Détracteurs** : note ∈ {1, 2}
- **Satisfaction nette (NPS-like)** = % Promoteurs − % Détracteurs
- **Gravité Haute** : note=1 + mots-clés forts, ou note=2 pour Bug/Financier/Annulation

In [ ]:
if N == 0:
    raise ValueError("Aucun avis : impossible de calculer les KPIs.")

note_moyenne = df["note"].mean()
note_mediane = df["note"].median()
promoteurs = 100.0 * df["note"].isin([4, 5]).sum() / N
detracteurs = 100.0 * df["note"].isin([1, 2]).sum() / N
passifs = 100.0 * (df["note"] == 3).sum() / N
nps_like = round(promoteurs - detracteurs, 1)
pct_haute = 100.0 * (df["Gravité"] == "Haute").sum() / N
pct_autre = 100.0 * (df["Catégorie"] == "Autre").sum() / N

print(f"{'=' * 60}")
print(f"{'KPIs GLOBAUX':^60}")
print(f"{'=' * 60}")
print(f"Nombre total d'avis        : {N}")
print(f"Note moyenne               : {note_moyenne:.2f} / 5")
print(f"Note médiane               : {note_mediane:.0f} / 5")
print(f"Promoteurs (4-5★)          : {promoteurs:.1f}%")
print(f"Détracteurs (1-2★)         : {detracteurs:.1f}%")
print(f"Passifs (3★)               : {passifs:.1f}%")
print(f"Satisfaction nette (NPS-like): {nps_like:+.1f}")
print(f"Gravité Haute              : {pct_haute:.1f}%")
print(f"Taux « Autre »             : {pct_autre:.1f}%")

# KPIs par source
print(f"\n{'KPIs PAR SOURCE':^60}")
print(f"{'-' * 60}")
for src in ["Google Play", "App Store", "Trustpilot"]:
    sub = df[df["source"] == src]
    n_s = len(sub)
    if n_s == 0:
        continue
    m = sub["note"].mean()
    med = sub["note"].median()
    prom = 100 * sub["note"].isin([4, 5]).sum() / n_s
    detr = 100 * sub["note"].isin([1, 2]).sum() / n_s
    nps_s = round(prom - detr, 1)
    print(f"{src:20s} | n={n_s:4d} | moy={m:.2f} | méd={med:.0f} | NPS-like={nps_s:+.1f}")

## 5. Distributions univariées

Histogrammes / barres : **notes**, **catégories** (finale = Ollama), **gravité**, **sources**.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

# Notes (échelle 1 à 5)
note_counts = df["note"].value_counts().reindex(ORDRE_NOTES, fill_value=0)
ax = axes[0, 0]
bars = ax.bar(
    note_counts.index.astype(str), note_counts.values, color="steelblue", edgecolor="white"
)
ax.set_title("Distribution des notes")
ax.set_xlabel("Note")
ax.set_ylabel("Nombre d'avis")
for bar, v in zip(bars, note_counts.values, strict=False):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 2, str(v), ha="center", va="bottom", fontsize=9)

# Catégories (finale = Ollama)
cat_counts = df["Catégorie"].value_counts()
ax = axes[0, 1]
cat_counts.plot.barh(ax=ax, color="teal", edgecolor="white")
ax.set_title("Distribution des catégories (Ollama)")
ax.set_xlabel("Nombre d'avis")
ax.invert_yaxis()

# Gravité
grav_counts = df["Gravité"].value_counts().reindex(ORDRE_GRAVITE, fill_value=0)
ax = axes[1, 0]
colors_grav = {"Haute": "tomato", "Moyenne": "orange", "Basse": "mediumseagreen"}
ax.bar(
    grav_counts.index,
    grav_counts.values,
    color=[colors_grav[g] for g in grav_counts.index],
    edgecolor="white",
)
ax.set_title("Distribution de la gravité")
ax.set_ylabel("Nombre d'avis")
for bar, v in zip(ax.patches, grav_counts.values, strict=False):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 2, str(v), ha="center", va="bottom", fontsize=9)

# Sources
src_counts = df["source"].value_counts()
ax = axes[1, 1]
ax.pie(
    src_counts.values,
    labels=src_counts.index,
    autopct="%1.1f%%",
    colors=[COULEURS_SRC.get(s, "gray") for s in src_counts.index],
    startangle=90,
)
ax.set_title("Répartition par source")

plt.tight_layout()
plt.show()

## 6. Analyse temporelle

Agrégation par **mois calendaire** (année-mois). Attention : les avis reflètent la date de
publication, pas la date d'utilisation. Le volume mensuel varie selon la couverture du scraping.

In [ ]:
df_time = df.dropna(subset=["date"]).copy()
df_time["annee_mois"] = df_time["date"].dt.to_period("M").astype(str)

par_mois = df_time.groupby("annee_mois", as_index=False).size()
par_mois = par_mois.rename(columns={"size": "nb_avis"})

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(par_mois["annee_mois"], par_mois["nb_avis"], marker="o", linewidth=2, markersize=6)
ax.set_title(
    f"Nombre d'avis par mois ({df_time['date'].dt.year.min()}–{df_time['date'].dt.year.max()})"
)
ax.set_xlabel("Mois")
ax.set_ylabel("Nombre d'avis")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(
    f"Mois max : {par_mois.loc[par_mois['nb_avis'].idxmax(), 'annee_mois']} "
    f"({par_mois['nb_avis'].max()} avis)"
)
print(
    f"Mois min : {par_mois.loc[par_mois['nb_avis'].idxmin(), 'annee_mois']} "
    f"({par_mois['nb_avis'].min()} avis)"
)

## 6b. Tendance de la note par source

Le trend agrégé mélange trois sources avec des biais différents. On décompose ici la tendance
**par source** pour isoler le signal réel de la satisfaction des utilisateurs.

In [ ]:
df_tr = df.dropna(subset=["date", "note"]).copy()
df_tr["mois"] = df_tr["date"].dt.to_period("M").astype(str)

pivot_note_src = df_tr.groupby(["mois", "source"])["note"].agg(["mean", "count"]).reset_index()
pivot_note_src.columns = ["mois", "source", "note_moy", "n"]

fig, ax = plt.subplots(figsize=(12, 5))
for src, color in COULEURS_SRC.items():
    grp = pivot_note_src[pivot_note_src["source"] == src].copy()
    grp_ok = grp[grp["n"] >= 5]  # Stabilité : minimum 5 avis/mois
    if len(grp_ok) < 3:
        continue
    ax.plot(
        grp_ok["mois"],
        grp_ok["note_moy"],
        marker="o",
        label=src,
        color=color,
        linewidth=2,
        markersize=5,
    )

ax.axhline(
    note_moyenne,
    color="gray",
    linestyle="--",
    alpha=0.6,
    label=f"Moyenne globale ({note_moyenne:.2f})",
)
ax.set_title("Tendance mensuelle de la note par source (≥ 5 avis/mois)")
ax.set_xlabel("Mois")
ax.set_ylabel("Note moyenne")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Volume empilé par source
pivot_vol_src = df_tr.groupby(["mois", "source"]).size().unstack(fill_value=0)
pivot_vol_src = pivot_vol_src[[s for s in COULEURS_SRC if s in pivot_vol_src.columns]]
fig, ax = plt.subplots(figsize=(12, 4))
pivot_vol_src.plot.area(ax=ax, color=[COULEURS_SRC[s] for s in pivot_vol_src.columns], alpha=0.7)
ax.set_title("Volume mensuel d'avis par source")
ax.set_ylabel("Nombre d'avis")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. Catégorisation : mots-clés vs IA (Ollama)

Le pipeline applique deux catégoriseurs en séquence :
1. **Mots-clés** : règles lexicales rapides (colonne `Catégorie_mots_cles`)
2. **Ollama gemma4:31b** : validation / correction par LLM (colonne `Catégorie_ollama` = `Catégorie`)

Cette section compare les deux approches pour quantifier l'apport de l'IA et identifier
les zones de faiblesse des mots-clés.

In [ ]:
kw = df["Catégorie_mots_cles"]
ol = df["Catégorie_ollama"]

accord = int((kw == ol).sum())
desaccord = int((kw != ol).sum())
taux_accord = 100 * accord / N

# Cohen's kappa (sans dépendance sklearn)
p_o = (kw == ol).mean()
cats_all = sorted(set(kw) | set(ol))
p_e = sum((kw == c).mean() * (ol == c).mean() for c in cats_all)
kappa = (p_o - p_e) / (1 - p_e) if p_e < 1 else 0.0

print(f"Accord mots-clés ↔ Ollama : {accord}/{N} ({taux_accord:.1f}%)")
print(f"Désaccords (corrections IA) : {desaccord} avis ({100 * desaccord / N:.1f}%)")
print(f"Cohen κ = {kappa:.3f}")
if kappa >= 0.8:
    print("  → Accord presque parfait")
elif kappa >= 0.6:
    print("  → Accord substantiel")
elif kappa >= 0.4:
    print("  → Accord modéré")
else:
    print("  → Accord faible — l'IA corrige significativement les mots-clés")

# Matrice de confusion
cats = sorted(df["Catégorie"].unique())
cm = pd.crosstab(kw, ol, margins=True, margins_name="Total")
print("\nMatrice de confusion (lignes = mots-clés, colonnes = Ollama) :")
display(cm)

# Heatmap
cm_core = pd.crosstab(kw, ol).reindex(index=cats, columns=cats, fill_value=0)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_core, annot=True, fmt="d", cmap="Blues", ax=ax, linewidths=0.5)
ax.set_xlabel("Catégorie Ollama (finale)")
ax.set_ylabel("Catégorie mots-clés")
ax.set_title(f"Confusion mots-clés → Ollama (accord {taux_accord:.0f}%, κ = {kappa:.2f})")
plt.tight_layout()
plt.show()

# Top des corrections Ollama
corrections = df[kw != ol][["texte", "note", "Catégorie_mots_cles", "Catégorie_ollama"]].copy()
corr_summary = (
    corrections.groupby(["Catégorie_mots_cles", "Catégorie_ollama"])
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)
print(f"\nTop corrections Ollama ({len(corrections)} avis reclassés) :")
display(corr_summary.head(15))

# Précision mots-clés par catégorie (vs Ollama comme référence)
cat_acc = pd.DataFrame({"n_mots_cles": kw.value_counts(), "n_ollama": ol.value_counts()})
cat_acc = cat_acc.fillna(0).astype(int)
cat_acc["accord"] = [int((kw[kw == c] == ol[kw == c]).sum()) for c in cat_acc.index]
cat_acc["precision_kw_%"] = (100 * cat_acc["accord"] / cat_acc["n_mots_cles"]).round(1)
cat_acc = cat_acc.sort_values("precision_kw_%")
print("\nPrécision mots-clés par catégorie (vs Ollama comme référence) :")
display(cat_acc)

## 8. Tableau croisé Catégorie × Gravité

- **Effectifs** : combinaisons observées.
- **% par ligne** : profil de gravité de chaque catégorie (somme = 100%).
- **Heatmap** : repérer visuellement les catégories à forte gravité.

In [ ]:
ct = pd.crosstab(df["Catégorie"], df["Gravité"], margins=True, margins_name="Total")
display(ct)

# % par ligne
ct_body = pd.crosstab(df["Catégorie"], df["Gravité"])
pct_ligne = (ct_body.div(ct_body.sum(axis=1), axis=0) * 100).round(1)
pct_ligne = pct_ligne.reindex(columns=ORDRE_GRAVITE, fill_value=0)
pct_ligne["Total ligne"] = ct_body.sum(axis=1)

print("\n% par ligne :")
display(pct_ligne)

# Heatmap
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    pct_ligne[ORDRE_GRAVITE],
    annot=True,
    fmt=".0f",
    cmap="YlOrRd",
    ax=ax,
    linewidths=0.5,
    cbar_kws={"label": "%"},
)
ax.set_title("% Gravité par catégorie")
plt.tight_layout()
plt.show()

## 9. Note selon la catégorie

Boîtes à moustaches : distribution des **notes (1–5)** par catégorie (triée par volume).
La médiane (trait central) est le meilleur indicateur pour des données ordinales.

In [ ]:
ordre_cat = df["Catégorie"].value_counts().index.tolist()
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="Catégorie", y="note", order=ordre_cat, palette="Set2")
plt.xticks(rotation=30, ha="right")
plt.title("Distribution des notes par catégorie")
plt.ylabel("Note (1–5)")
plt.tight_layout()
plt.show()

# Statistiques par catégorie
stats_cat = df.groupby("Catégorie")["note"].agg(["count", "mean", "median", "std"]).round(2)
stats_cat = stats_cat.sort_values("count", ascending=False)
display(stats_cat)

## 10. Longueur des commentaires (proxy d'effort / détail)

- `longueur_texte` = nombre de **mots** (calculé par le pipeline)
- `n_caracteres` = nombre de **caractères**

Corrélation de **Spearman** (ρ) entre la note et la longueur : les avis négatifs sont-ils plus détaillés ?

In [ ]:
from scipy.stats import spearmanr

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution longueur (mots)
axes[0].hist(df["longueur_texte"], bins=40, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(
    df["longueur_texte"].median(),
    color="tomato",
    linestyle="--",
    label=f"Médiane = {df['longueur_texte'].median():.0f} mots",
)
axes[0].set_title("Distribution de la longueur (mots)")
axes[0].set_xlabel("Nombre de mots")
axes[0].set_ylabel("Fréquence")
axes[0].legend()

# Longueur par note (boxplot)
sns.boxplot(
    data=df, x="note", y="longueur_texte", order=ORDRE_NOTES, ax=axes[1], palette="coolwarm"
)
axes[1].set_title("Longueur par note")
axes[1].set_xlabel("Note")
axes[1].set_ylabel("Nombre de mots")

plt.tight_layout()
plt.show()

# Corrélation de Spearman
rho_mots, p_mots = spearmanr(df["note"], df["longueur_texte"], nan_policy="omit")
rho_chars, p_chars = spearmanr(df["note"], df["n_caracteres"], nan_policy="omit")
print(f"Spearman ρ (note vs mots)       = {rho_mots:.3f}, p = {p_mots:.4f}")
print(f"Spearman ρ (note vs caractères) = {rho_chars:.3f}, p = {p_chars:.4f}")
if p_mots < ALPHA:
    direction = "négatif" if rho_mots < 0 else "positif"
    print(
        f"  → Corrélation significative ({direction}) : les avis {'négatifs' if rho_mots < 0 else 'positifs'} "
        f"sont {'plus longs' if rho_mots < 0 else 'plus courts'}."
    )
else:
    print("  → Pas de corrélation significative.")

## 11. Courbe de Pareto (catégories)

Les **k** premières catégories cumulent quelle part du volume ?
La règle des 80/20 s'applique-t-elle ?

In [ ]:
cat_sorted = df["Catégorie"].value_counts().sort_values(ascending=False)
cum_pct = (100 * cat_sorted.cumsum() / N).round(1)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(range(len(cat_sorted)), cat_sorted.values, color="teal", edgecolor="white", label="Volume")
ax1.set_xticks(range(len(cat_sorted)))
ax1.set_xticklabels(cat_sorted.index, rotation=30, ha="right")
ax1.set_ylabel("Nombre d'avis")

ax2 = ax1.twinx()
ax2.plot(
    range(len(cum_pct)),
    cum_pct.values,
    marker="D",
    color="tomato",
    linewidth=2,
    markersize=6,
    label="Cumul %",
)
ax2.axhline(80, color="gray", linestyle="--", alpha=0.5, label="Seuil 80%")
ax2.set_ylabel("% cumulé")
ax2.set_ylim(0, 105)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")
ax1.set_title("Pareto des catégories")
plt.tight_layout()
plt.show()

# Nombre de catégories pour atteindre 80%
n_80 = int((cum_pct <= 80).sum()) + 1
print(f"{n_80} catégories couvrent ≥ 80% du volume.")

## 12. Catégorie secondaire (multi-thèmes)

Le pipeline peut attacher une **deuxième** catégorie quand le texte touche à plusieurs thèmes.
Quelle proportion d'avis est multi-catégorie ? Quelles paires reviennent le plus ?

In [ ]:
has_sec = df["Catégorie_secondaire"].str.strip() != ""
n_multi = int(has_sec.sum())
pct_multi = 100.0 * n_multi / N
print(f"Avis multi-catégorie : {n_multi} / {N} ({pct_multi:.1f}%)")

if n_multi > 0:
    pairs = (
        df[has_sec]
        .groupby(["Catégorie", "Catégorie_secondaire"])
        .size()
        .reset_index(name="Effectif")
        .sort_values("Effectif", ascending=False)
    )
    print("\nTop 15 paires primaire → secondaire :")
    display(pairs.head(15))

    # Heatmap co-occurrences
    top_sec = pairs["Catégorie_secondaire"].value_counts().head(7).index.tolist()
    heatmap_data = (
        df[has_sec & df["Catégorie_secondaire"].isin(top_sec)]
        .groupby(["Catégorie", "Catégorie_secondaire"])
        .size()
        .unstack(fill_value=0)
    )
    if not heatmap_data.empty:
        fig, ax = plt.subplots(figsize=(9, 5))
        sns.heatmap(heatmap_data, annot=True, fmt="d", cmap="YlGnBu", ax=ax, linewidths=0.5)
        ax.set_title("Co-occurrences Catégorie primaire × secondaire")
        plt.tight_layout()
        plt.show()

## 13. Comparaison inter-sources

Les trois sources attirent-elles les **mêmes profils d'avis** ?

- **Kruskal-Wallis** : test non-paramétrique pour comparer les distributions de notes
- **Dunn post-hoc** : comparaisons par paires avec correction de Bonferroni
- **Chi² Catégorie × Source** + Cramér V : indépendance des thèmes et de la source

In [ ]:
from scipy.stats import chi2_contingency, kruskal

# --- Kruskal-Wallis : notes par source ---
groups = [g["note"].dropna().values for _, g in df.groupby("source")]
stat_kw, p_kw = kruskal(*groups)
print(f"Kruskal-Wallis H = {stat_kw:.3f}, p = {p_kw:.4f}")
print(
    "→",
    "Distributions significativement différentes"
    if p_kw < ALPHA
    else "Pas de différence significative",
)

# --- Dunn post-hoc (si KW significatif) ---
if p_kw < ALPHA:
    try:
        import scikit_posthocs as sp

        dunn = sp.posthoc_dunn(
            df.dropna(subset=["note"]), val_col="note", group_col="source", p_adjust="bonferroni"
        )
        print("\nTest de Dunn (p-values ajustées Bonferroni) :")
        display(dunn.round(4))

        fig, ax = plt.subplots(figsize=(5, 4))
        mask = dunn > ALPHA
        sns.heatmap(
            dunn,
            annot=True,
            fmt=".4f",
            cmap="RdYlGn",
            mask=mask,
            ax=ax,
            vmin=0,
            vmax=0.05,
            linewidths=0.5,
        )
        ax.set_title(f"Dunn post-hoc (α = {ALPHA})")
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("⚠ scikit_posthocs non installé — test de Dunn omis")

# --- Chi² Catégorie × Source ---
print("\n--- Chi² Catégorie × Source ---")
ct_src = pd.crosstab(df["Catégorie"], df["source"])
chi2_s, p_s, dof_s, exp_s = chi2_contingency(ct_src)
n_s = int(ct_src.values.sum())
r_s, c_s = ct_src.shape
cramer_s = np.sqrt(chi2_s / (n_s * min(r_s - 1, c_s - 1))) if n_s > 0 else float("nan")
print(f"Chi² = {chi2_s:.2f}, ddl = {dof_s}, p = {p_s:.4f}")
print(f"Cramér V = {cramer_s:.3f}")
low_exp_s = int((exp_s < 5).sum())
if low_exp_s:
    print(f"⚠ {low_exp_s} cellule(s) avec effectif attendu < 5")

# Heatmap Catégorie × Source
fig, ax = plt.subplots(figsize=(8, 5))
pct_src = ct_src.div(ct_src.sum(axis=0), axis=1) * 100
sns.heatmap(
    pct_src,
    annot=True,
    fmt=".0f",
    cmap="YlGnBu",
    ax=ax,
    linewidths=0.5,
    cbar_kws={"label": "% de la source"},
)
ax.set_title(f"% catégorie par source (Cramér V = {cramer_s:.2f})")
plt.tight_layout()
plt.show()

### Interprétation de la comparaison inter-sources

- **Kruskal-Wallis** confirme que les distributions de notes diffèrent significativement entre sources.
- **Dunn post-hoc** identifie les paires source-source dont les distributions divergent (après correction Bonferroni).
- **Cramér V (Catégorie × Source)** mesure la force de l'association entre catégories et sources : certaines plaintes sont source-spécifiques.

**Ce que cela signifie pour l'équipe produit** :
- Le score de satisfaction Abritel doit être mesuré **sur Google Play uniquement** pour un signal équilibré.
- Trustpilot et App Store sont utiles pour la **détection de thèmes de plaintes**, pas pour mesurer une tendance de satisfaction.
- Un pic d'avis Trustpilot sur une période = suspect (campagne, incident viral) — pas une dégradation produit.

## 14. Analyse textuelle (fréquences de mots)

Mots les plus fréquents (stopwords **français + anglais** retirés), bigrammes, et **lift
par catégorie** (surreprésentation d'un mot dans une catégorie vs le corpus global).

> **Lemmatisation** : les tokens sont lemmatisés avec `simplemma` (français) pour regrouper
> les variantes morphologiques (« problème » / « problèmes » → « problème »).

In [ ]:
import re
from collections import Counter
from itertools import pairwise

from simplemma import lemmatize

_STOPWORDS_FR = {
    "de",
    "du",
    "des",
    "le",
    "la",
    "les",
    "un",
    "une",
    "et",
    "en",
    "est",
    "que",
    "qui",
    "dans",
    "pour",
    "pas",
    "sur",
    "au",
    "aux",
    "avec",
    "son",
    "ses",
    "mon",
    "mes",
    "ce",
    "cette",
    "ces",
    "mais",
    "ou",
    "par",
    "ne",
    "plus",
    "tres",
    "tout",
    "fait",
    "bien",
    "comme",
    "aussi",
    "bon",
    "elle",
    "ete",
    "ont",
    "nous",
    "vous",
    "ils",
    "avoir",
    "etre",
    "faire",
    "dire",
    "peut",
    "quand",
    "sont",
    "moi",
    "lui",
}
_STOPWORDS_EN = {
    "the",
    "and",
    "is",
    "it",
    "to",
    "of",
    "in",
    "for",
    "on",
    "that",
    "this",
    "with",
    "was",
    "are",
    "not",
    "but",
    "have",
    "has",
    "had",
    "you",
    "your",
    "very",
    "all",
    "been",
    "from",
    "they",
    "will",
    "would",
    "there",
    "their",
    "can",
    "just",
}
_STOPWORDS = _STOPWORDS_FR | _STOPWORDS_EN | {"app", "application", "abritel"}


def _tokenize(text: str) -> list[str]:
    tokens = re.findall(r"[a-zàâäéèêëïîôùûüçœæ]+", text.lower())
    return [lemmatize(w, lang="fr") for w in tokens if len(w) > 2 and w not in _STOPWORDS]


all_tokens = df["texte"].apply(_tokenize)

# Top 25 mots globaux
global_counter = Counter()
for tokens in all_tokens:
    global_counter.update(tokens)

top_words = global_counter.most_common(25)
fig, ax = plt.subplots(figsize=(10, 5))
words, counts = zip(*top_words, strict=True)
ax.barh(list(reversed(words)), list(reversed(counts)), color="steelblue")
ax.set_title("Top 25 mots les plus fréquents (lemmatisés, hors stopwords)")
ax.set_xlabel("Fréquence")
plt.tight_layout()
plt.show()

# Lift par catégorie (surreprésentation)
MIN_CAT_TOKENS = 30
MIN_W_CAT = 5
total_tokens = sum(global_counter.values())
p_w_global = {w: c / total_tokens for w, c in global_counter.items() if c >= 10}

print("\nTop lift par catégorie (mots surreprésentés vs corpus global) :")
for cat in sorted(df["Catégorie"].unique()):
    if cat == "Autre":
        continue
    cat_tokens = Counter()
    for tokens in all_tokens[df["Catégorie"] == cat]:
        cat_tokens.update(tokens)
    n_cat = sum(cat_tokens.values())
    if n_cat < MIN_CAT_TOKENS:
        continue
    lifts = {}
    for w, c in cat_tokens.items():
        pg = p_w_global.get(w)
        if pg and c >= MIN_W_CAT:
            lifts[w] = (c / n_cat) / pg
    top_lift = sorted(lifts.items(), key=lambda x: -x[1])[:5]
    if top_lift:
        top_str = ", ".join(f"{w} ({lift:.1f}×)" for w, lift in top_lift)
        print(f"  {cat:30s} → {top_str}")

# Bigrammes
bigram_counter = Counter()
for tokens in all_tokens:
    for a, b in pairwise(tokens):
        bigram_counter[(a, b)] += 1

top_bigrams = bigram_counter.most_common(15)
print("\nTop 15 bigrammes :")
for (a, b), cnt in top_bigrams:
    print(f"  {a} {b:20s} — {cnt}")

## 15. Test d'indépendance Catégorie × Gravité (χ²) + ASR + Fisher

Un test du **chi-deux** vérifie si la gravité dépend de la catégorie.
Les **résidus standardisés ajustés** (ASR) identifient les combinaisons sur- ou sous-représentées.

> **Fisher exact test** : quand des cellules ont un effectif attendu < 5, le test exact de Fisher
> est utilisé en complément (plus fiable pour les petits échantillons).

> **Mise en garde** : la Gravité est calculée depuis la note ET la catégorie (note=2 +
> Bug/Financier/Annulation → Haute). Une partie de l'association est **construite**, pas
> observée indépendamment. Voir la section **15b** pour `Gravité_texte`, un signal indépendant.

In [ ]:
from scipy.stats import chi2_contingency, fisher_exact

ct_test = pd.crosstab(df["Catégorie"], df["Gravité"])
chi2, p_chi2, dof, expected = chi2_contingency(ct_test)
n_ct = int(ct_test.values.sum())
r_ct, c_ct = ct_test.shape
cramer_v = np.sqrt(chi2 / (n_ct * min(r_ct - 1, c_ct - 1))) if n_ct > 0 else float("nan")
print(f"Chi² = {chi2:.2f},  ddl = {dof},  p-value = {p_chi2:.4f}")
print(f"Cramér V (taille d'effet) = {cramer_v:.3f}")
low_exp = int((expected < 5).sum())
if low_exp:
    print(f"⚠ {low_exp} cellule(s) avec effectif attendu < 5 — interpréter avec prudence")

# Fisher exact test par paire catégorie × gravité (quand cellules < 5)
if low_exp > 0:
    print("\n--- Tests de Fisher exacts (paires avec effectif attendu < 5) ---")
    for i, cat in enumerate(ct_test.index):
        for j, grav in enumerate(ct_test.columns):
            if expected[i, j] < 5:
                # Table 2×2 : cette cellule vs le reste
                a = ct_test.values[i, j]
                b = ct_test.values[i, :].sum() - a
                c = ct_test.values[:, j].sum() - a
                d = n_ct - a - b - c
                _, p_fisher = fisher_exact([[a, b], [c, d]])
                sig = (
                    "***"
                    if p_fisher < 0.001
                    else ("**" if p_fisher < 0.01 else ("*" if p_fisher < 0.05 else "n.s."))
                )
                print(
                    f"  {cat} × {grav} : obs={a}, att={expected[i, j]:.1f}, Fisher p={p_fisher:.4f} {sig}"
                )

# Résidus standardisés ajustés (ASR)
row_totals = ct_test.sum(axis=1).values
col_totals = ct_test.sum(axis=0).values
n_total = ct_test.values.sum()

asr = np.zeros_like(ct_test.values, dtype=float)
for i in range(ct_test.shape[0]):
    for j in range(ct_test.shape[1]):
        e_ij = expected[i, j]
        denom = np.sqrt(e_ij * (1 - row_totals[i] / n_total) * (1 - col_totals[j] / n_total))
        asr[i, j] = (ct_test.values[i, j] - e_ij) / denom if denom > 0 else 0

asr_df = pd.DataFrame(asr, index=ct_test.index, columns=ct_test.columns)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    asr_df,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    ax=ax,
    linewidths=0.5,
    cbar_kws={"label": "ASR"},
)
ax.set_title(f"Résidus standardisés ajustés — Catégorie × Gravité (Cramér V = {cramer_v:.2f})")
plt.tight_layout()
plt.show()

# Associations significatives
print("\nAssociations significatives (|ASR| > 1.96) :")
for cat in asr_df.index:
    for grav in asr_df.columns:
        val = asr_df.loc[cat, grav]
        if abs(val) > 1.96:
            direction = "SUR-représenté" if val > 0 else "sous-représenté"
            print(f"  {cat} × {grav} : ASR = {val:+.2f} → {direction}")

## 15b. Gravité indépendante (texte seul) — correction de la tautologie

`Gravité_texte` est calculée **uniquement à partir du texte** (mots de colère, urgence légale,
vocabulaire négatif), sans utiliser la note ni la catégorie. Cela permet de vérifier si
l'association Catégorie × Gravité reste significative avec un signal indépendant.

**Comparaison** : `Gravité` (note + catégorie + texte) vs `Gravité_texte` (texte seul).

In [ ]:
# Vérifier que Gravité_texte est présente (nécessite re-run du pipeline)
if "Gravité_texte" not in df.columns:
    from abritel.categorisation import evaluer_gravite_texte

    df["Gravité_texte"] = df["texte"].map(evaluer_gravite_texte)

# Comparaison Gravité vs Gravité_texte
accord_grav = (df["Gravité"] == df["Gravité_texte"]).sum()
print(f"Accord Gravité ↔ Gravité_texte : {accord_grav}/{N} ({100 * accord_grav / N:.1f}%)")

# Crosstab Gravité × Gravité_texte
ct_grav = pd.crosstab(df["Gravité"], df["Gravité_texte"], margins=True, margins_name="Total")
print("\nMatrice de confusion (lignes = Gravité standard, colonnes = Gravité texte seul) :")
display(ct_grav)

# Distribution Gravité_texte
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, titre in [
    (axes[0], "Gravité", "Gravité (note + catégorie + texte)"),
    (axes[1], "Gravité_texte", "Gravité_texte (texte seul)"),
]:
    counts = df[col].value_counts().reindex(ORDRE_GRAVITE, fill_value=0)
    colors = [colors_grav[g] for g in counts.index]
    ax.bar(counts.index, counts.values, color=colors, edgecolor="white")
    ax.set_title(titre)
    ax.set_ylabel("Nombre d'avis")
    for bar, v in zip(ax.patches, counts.values, strict=False):
        ax.text(
            bar.get_x() + bar.get_width() / 2, v + 2, str(v), ha="center", va="bottom", fontsize=9
        )

plt.tight_layout()
plt.show()

# Chi² Catégorie × Gravité_texte (test INDÉPENDANT de la tautologie)
ct_indep = pd.crosstab(df["Catégorie"], df["Gravité_texte"])
chi2_i, p_i, dof_i, exp_i = chi2_contingency(ct_indep)
n_i = int(ct_indep.values.sum())
r_i, c_i = ct_indep.shape
cramer_i = np.sqrt(chi2_i / (n_i * min(r_i - 1, c_i - 1))) if n_i > 0 else float("nan")
print(f"\nChi² Catégorie × Gravité_texte = {chi2_i:.2f}, ddl = {dof_i}, p = {p_i:.4f}")
print(f"Cramér V (indépendant) = {cramer_i:.3f}")
print(f"Cramér V (standard, avec tautologie) = {cramer_v:.3f}")
print(f"→ Écart : {abs(cramer_v - cramer_i):.3f} — la part tautologique du Cramér V original")

low_exp_i = int((exp_i < 5).sum())
if low_exp_i:
    print(f"⚠ {low_exp_i} cellule(s) avec effectif attendu < 5")

# Heatmap Catégorie × Gravité_texte
pct_gt = ct_indep.div(ct_indep.sum(axis=1), axis=0) * 100
pct_gt = pct_gt.reindex(columns=ORDRE_GRAVITE, fill_value=0)
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    pct_gt, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax, linewidths=0.5, cbar_kws={"label": "%"}
)
ax.set_title(f"% Gravité_texte par catégorie (Cramér V = {cramer_i:.2f}, indépendant)")
plt.tight_layout()
plt.show()

## 16. Tendance temporelle et test de tendance

- **Moyenne mobile 30 jours** de la note : la satisfaction s'améliore-t-elle ?
- **Kendall τ** sur les moyennes mensuelles : test de tendance monotone.
- **Volume mensuel par gravité** : évolution de la composition.

In [ ]:
from scipy.stats import kendalltau

# Moyenne mobile 30 jours
df_sorted = df.dropna(subset=["date", "note"]).sort_values("date").copy()
df_sorted = df_sorted.set_index("date")
rolling_mean = df_sorted["note"].rolling("30D", min_periods=5).mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(rolling_mean.index, rolling_mean.values, color="steelblue", linewidth=2)
ax.axhline(
    df["note"].mean(),
    color="gray",
    linestyle="--",
    alpha=0.6,
    label=f"Moyenne globale ({df['note'].mean():.2f})",
)
ax.set_title("Moyenne mobile 30 jours de la note")
ax.set_ylabel("Note moyenne")
ax.legend()
plt.tight_layout()
plt.show()

# Kendall τ sur moyennes mensuelles
df_time2 = df.dropna(subset=["date", "note"]).copy()
df_time2["mois"] = df_time2["date"].dt.to_period("M").astype(str)
monthly_mean = df_time2.groupby("mois", sort=True)["note"].mean()
tau, p_tau = kendalltau(range(len(monthly_mean)), monthly_mean.values)
print(f"Kendall τ = {tau:.3f}, p = {p_tau:.4f} (n = {len(monthly_mean)} mois)")
if p_tau < ALPHA:
    direction = "hausse" if tau > 0 else "baisse"
    print(f"  → Tendance significative à la {direction}")
else:
    print("  → Pas de tendance monotone significative")

# Volume par gravité
df_time3 = df.dropna(subset=["date"]).copy()
df_time3["mois"] = df_time3["date"].dt.to_period("M").astype(str)
pivot_grav = df_time3.groupby(["mois", "Gravité"]).size().unstack(fill_value=0)
pivot_grav = pivot_grav.reindex(columns=ORDRE_GRAVITE, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 4))
pivot_grav.plot.area(ax=ax, color=["tomato", "orange", "mediumseagreen"], alpha=0.7)
ax.set_title("Volume mensuel par gravité")
ax.set_ylabel("Nombre d'avis")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 16b. Évolution temporelle par catégorie

Quelles catégories de plaintes **progressent ou déclinent** sur la période ?
Détection de tendances actionnables via Kendall τ par catégorie.

In [ ]:
from scipy.stats import kendalltau as _kt

df_ct = df.dropna(subset=["date", "note"]).copy()
df_ct["mois"] = df_ct["date"].dt.to_period("M").astype(str)

# Volume mensuel par catégorie (hors Autre)
cats_analyse = [c for c in df["Catégorie"].unique() if c != "Autre"]
pivot_cat = df_ct.groupby(["mois", "Catégorie"]).size().unstack(fill_value=0)
pivot_cat = pivot_cat[[c for c in cats_analyse if c in pivot_cat.columns]]
all_months = sorted(pivot_cat.index.unique())

# Kendall tau par catégorie
print("Tendance par catégorie (Kendall τ) :")
trends = []
for cat in pivot_cat.columns:
    series = pivot_cat[cat].reindex(all_months, fill_value=0)
    if series.sum() < 10:
        continue
    t, p = _kt(range(len(series)), series.values)
    sig = "↑" if (t > 0 and p < ALPHA) else ("↓" if (t < 0 and p < ALPHA) else "—")
    trends.append((cat, t, p, sig))
    print(f"  {cat:30s} τ = {t:+.3f}, p = {p:.3f} {sig}")

# Graphique des parts mensuelles (%)
pivot_pct = pivot_cat.div(pivot_cat.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(12, 5))
for cat in pivot_cat.columns:
    ax.plot(pivot_pct.index, pivot_pct[cat], marker=".", label=cat, linewidth=1.5)
ax.set_title("Parts mensuelles par catégorie (hors Autre)")
ax.set_ylabel("% du volume mensuel")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 17. Deep dive : 1★ vs 5★

Comparaison des catégories, de la longueur de texte et échantillons qualitatifs
aux deux extrêmes de la satisfaction.

In [ ]:
df_1star = df[df["note"] == 1]
df_5star = df[df["note"] == 5]

# Catégories comparées
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, sub, titre, color in [
    (axes[0], df_1star, f"1★ (n={len(df_1star)})", "tomato"),
    (axes[1], df_5star, f"5★ (n={len(df_5star)})", "seagreen"),
]:
    cat_c = sub["Catégorie"].value_counts()
    ax.barh(cat_c.index[::-1], cat_c.values[::-1], color=color, edgecolor="white")
    ax.set_title(titre)
    ax.set_xlabel("Nombre d'avis")
fig.suptitle("Catégories — 1★ vs 5★", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Longueur moyenne par extrême
print("Longueur moyenne (mots) :")
print(
    f"  1★ : {df_1star['longueur_texte'].mean():.1f} mots (médiane {df_1star['longueur_texte'].median():.0f})"
)
print(
    f"  5★ : {df_5star['longueur_texte'].mean():.1f} mots (médiane {df_5star['longueur_texte'].median():.0f})"
)

# Exemples
print("\nÉchantillons 1★ (5 avis aléatoires) :")
for _, row in df_1star.sample(min(5, len(df_1star)), random_state=RANDOM_STATE).iterrows():
    print(f"  [{row['source']}] {str(row['texte'])[:150]}")

print("\nÉchantillons 5★ (5 avis aléatoires) :")
for _, row in df_5star.sample(min(5, len(df_5star)), random_state=RANDOM_STATE).iterrows():
    print(f"  [{row['source']}] {str(row['texte'])[:150]}")

## 17b. Deep dive — Catégorie « Autre » et sous-types

Les avis « Autre » sont ceux qu'aucun mot-clé ne couvre. Le pipeline les sous-type
via `Autre_type` :
- **positif court** : avis positif (note >= 4) et court (<=15 mots)
- **positif thématique** : avis positif (note >= 4) avec marqueurs de satisfaction détectés par mots-clés positifs
- **neutre** : avis sans signal clair
- **négatif non catégorisé** : avis négatif que les mots-clés n'ont pas capté (angle mort)

In [ ]:
autres = df[df["Catégorie"] == "Autre"].copy()
n_autres = len(autres)
print(f"Avis « Autre » : {n_autres} ({100 * n_autres / N:.1f}% du corpus)")
print()

# Distribution par sous-type
type_counts = autres["Autre_type"].value_counts()
print("Sous-types :")
for typ, cnt in type_counts.items():
    print(f"  {typ:30s} : {cnt:4d} ({100 * cnt / n_autres:.1f}%)")

# Notes par sous-type
sous_types = ["positif court", "positif thématique", "neutre", "négatif non catégorisé"]
sous_types_present = [s for s in sous_types if s in type_counts.index]
n_cols = len(sous_types_present)
if n_cols > 0:
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4), sharey=True)
    if n_cols == 1:
        axes = [axes]
    for ax, typ in zip(axes, sous_types_present, strict=True):
        sub = autres[autres["Autre_type"] == typ]
        if sub.empty:
            ax.set_title(f"{typ} (n=0)")
            continue
        counts = sub["note"].value_counts().reindex(ORDRE_NOTES, fill_value=0)
        ax.bar(counts.index.astype(str), counts.values, color="teal", edgecolor="white")
        ax.set_title(f"{typ}\n(n={len(sub)}, moy={sub['note'].mean():.2f})")
        ax.set_xlabel("Note")
    axes[0].set_ylabel("Nombre d'avis")
    fig.suptitle("Distribution des notes par sous-type « Autre »", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

# Exemples angle mort (négatifs non catégorisés)
neg_nc = autres[autres["Autre_type"] == "négatif non catégorisé"]
if not neg_nc.empty:
    print(f"\nAvis « négatif non catégorisé » ({len(neg_nc)} avis) — angle mort du système :")
    for _, row in neg_nc.iterrows():
        print(f"  [{row['source']}, {row['note']}★] {str(row['texte'])[:250]}")
else:
    print("\nAucun avis négatif non catégorisé.")

# Positifs thématiques
pos_th = autres[autres["Autre_type"] == "positif thématique"]
if not pos_th.empty:
    print(
        f"\nAvis « positif thématique » ({len(pos_th)} avis) — satisfaction détectée par mots-clés :"
    )
    for _, row in pos_th.sample(min(5, len(pos_th)), random_state=RANDOM_STATE).iterrows():
        print(f"  [{row['source']}, {row['note']}★] {str(row['texte'])[:250]}")

## 18. Analyse par version release

Chaque avis est rattaché à la version de l'app active à sa date de publication
(via `data/releases.csv`). On analyse l'évolution de la satisfaction et des catégories
de plaintes **par version** pour mesurer l'impact des mises à jour.

In [ ]:
# Charger les descriptions des releases
rel = pd.read_csv(RELEASES_PATH)

# Statistiques par version
ver_stats = (
    df.groupby("version_release")
    .agg(
        n_avis=("note", "size"),
        note_moy=("note", "mean"),
        note_med=("note", "median"),
        pct_haute=("Gravité", lambda s: 100 * (s == "Haute").sum() / len(s)),
        pct_autre=("Catégorie", lambda s: 100 * (s == "Autre").sum() / len(s)),
    )
    .round(2)
)

# Joindre la description de la release
ver_stats = ver_stats.join(rel.set_index("version")[["description", "plateforme"]], how="left")
# Trier par version (chronologique)
ver_stats = ver_stats.sort_index()
print("Statistiques par version release :")
display(ver_stats)

# Graphique volume + note par version
fig, ax1 = plt.subplots(figsize=(13, 5))
x = range(len(ver_stats))
ax1.bar(x, ver_stats["n_avis"], color="steelblue", alpha=0.7, label="Volume")
ax1.set_xticks(x)
ax1.set_xticklabels(ver_stats.index, rotation=45, ha="right", fontsize=9)
ax1.set_ylabel("Volume d'avis")
ax1.set_xlabel("Version")

ax2 = ax1.twinx()
ax2.plot(
    x,
    ver_stats["note_moy"],
    marker="o",
    color="tomato",
    linewidth=2,
    markersize=6,
    label="Note moyenne",
)
ax2.set_ylabel("Note moyenne")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
ax1.set_title("Volume et note moyenne par version release")
plt.tight_layout()
plt.show()

# Catégories principales par version (hors Autre) — top 5 versions par volume
top_versions = ver_stats.nlargest(5, "n_avis").index.tolist()
cats_hors_autre = [c for c in df["Catégorie"].unique() if c != "Autre"]

fig, axes = plt.subplots(1, len(top_versions), figsize=(16, 4), sharey=True)
for ax, ver in zip(axes, top_versions, strict=True):
    sub = df[(df["version_release"] == ver) & (df["Catégorie"] != "Autre")]
    if sub.empty:
        continue
    cat_c = sub["Catégorie"].value_counts()
    ax.barh(cat_c.index[::-1], cat_c.values[::-1], color="teal", edgecolor="white")
    desc = ver_stats.loc[ver, "description"] if "description" in ver_stats.columns else ""
    desc_short = str(desc)[:30] if pd.notna(desc) else ""
    ax.set_title(f"v{ver}\n({desc_short})", fontsize=9)
axes[0].set_xlabel("Nombre d'avis")
fig.suptitle("Catégories par version (top 5 par volume, hors Autre)", fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

## 19. Matrice de priorisation (Volume × % Gravité Haute)

Scatter plot croisant le **volume** d'avis (axe X) et le **% de gravité Haute** (axe Y).
Taille des bulles ∝ score composite (volume × %haute / 100).
Couleur = note moyenne (rouge = bas, vert = haut).

Double lecture : **global** (toutes sources) et **Google Play** (source de référence).

In [ ]:
def _scatter_prio(data, title):
    prio = (
        data.groupby("Catégorie")
        .agg(
            volume=("note", "size"),
            note_moyenne=("note", "mean"),
            pct_haute=("Gravité", lambda s: 100 * (s == "Haute").sum() / len(s)),
        )
        .round(2)
    )
    prio["score"] = (prio["volume"] * prio["pct_haute"] / 100).round(1)
    prio = prio.sort_values("score", ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    scatter = ax.scatter(
        prio["volume"],
        prio["pct_haute"],
        s=prio["score"] * 3 + 30,
        c=prio["note_moyenne"],
        cmap="RdYlGn",
        vmin=1,
        vmax=5,
        edgecolors="black",
        linewidth=0.5,
        alpha=0.8,
    )
    for cat, row in prio.iterrows():
        ax.annotate(
            cat,
            (row["volume"], row["pct_haute"]),
            textcoords="offset points",
            xytext=(8, 5),
            fontsize=9,
        )
    ax.axhline(prio["pct_haute"].median(), color="gray", linestyle=":", alpha=0.5)
    ax.axvline(prio["volume"].median(), color="gray", linestyle=":", alpha=0.5)
    ax.set_xlabel("Volume d'avis")
    ax.set_ylabel("% Gravité Haute")
    ax.set_title(title)
    plt.colorbar(scatter, label="Note moyenne")
    plt.tight_layout()
    plt.show()
    return prio


# Global
prio_all = _scatter_prio(df, "Matrice de priorisation — Toutes sources")
display(prio_all)

# Google Play uniquement (référence)
gp = df[df["source"] == "Google Play"]
prio_gp = _scatter_prio(gp, "Matrice de priorisation — Google Play uniquement")
display(prio_gp)

## 20. Exports pour Power BI

Fichiers dérivés dans `data/` :
- `synthese_categories.csv` — volume et cumul par catégorie
- `crosstab_categorie_gravite.csv` — tableau croisé
- `crosstab_categorie_source.csv` — tableau croisé
- `synthese_secondaire.csv` — paires primaire → secondaire
- `priorisation_categories.csv` — scores de priorisation
- `synthese_categories_reporting.csv` — métriques enrichies pour reporting

In [ ]:
out_dir = Path("data")
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Synthèse catégories
synthese = df["Catégorie"].value_counts().reset_index()
synthese.columns = ["Catégorie", "Effectif"]
synthese["pct_total"] = (100 * synthese["Effectif"] / N).round(2)
synthese["pct_cumule"] = synthese["pct_total"].cumsum().round(2)
synthese.to_csv(out_dir / "synthese_categories.csv", index=False, encoding="utf-8")
print(f"✓ synthese_categories.csv ({len(synthese)} lignes)")

# 2. Crosstab Catégorie × Gravité
ct_export_grav = pd.crosstab(df["Catégorie"], df["Gravité"])
ct_export_grav.to_csv(out_dir / "crosstab_categorie_gravite.csv", encoding="utf-8")
print(f"✓ crosstab_categorie_gravite.csv ({ct_export_grav.shape})")

# 3. Crosstab Catégorie × Source
ct_export_src = pd.crosstab(df["Catégorie"], df["source"])
ct_export_src.to_csv(out_dir / "crosstab_categorie_source.csv", encoding="utf-8")
print(f"✓ crosstab_categorie_source.csv ({ct_export_src.shape})")

# 4. Synthèse secondaire
has_sec_export = df["Catégorie_secondaire"].str.strip() != ""
pairs_export = (
    df[has_sec_export]
    .groupby(["Catégorie", "Catégorie_secondaire"])
    .size()
    .reset_index(name="Effectif")
    .sort_values("Effectif", ascending=False)
)
pairs_export.to_csv(out_dir / "synthese_secondaire.csv", index=False, encoding="utf-8")
print(f"✓ synthese_secondaire.csv ({len(pairs_export)} paires)")

# 5. Priorisation
prio_export = prio_all.copy()
prio_export["mediane_volume"] = prio_all["volume"].median()
prio_export["mediane_pct_haute"] = prio_all["pct_haute"].median()
prio_export.to_csv(out_dir / "priorisation_categories.csv", encoding="utf-8")
print(f"✓ priorisation_categories.csv ({len(prio_export)} lignes)")

# 6. Reporting enrichi
reporting = (
    df.groupby("Catégorie")
    .agg(
        Effectif=("note", "size"),
        pct_total=("note", lambda s: round(100 * len(s) / N, 2)),
        note_moyenne=("note", "mean"),
        note_mediane=("note", "median"),
        pct_haute=("Gravité", lambda s: round(100 * (s == "Haute").sum() / len(s), 1)),
        pct_1etoile=("note", lambda s: round(100 * (s == 1).sum() / len(s), 1)),
        longueur_moy_mots=("longueur_texte", "mean"),
    )
    .round(2)
)
reporting.index.name = "Catégorie_reporting"
reporting = reporting.sort_values("Effectif", ascending=False)
reporting.to_csv(out_dir / "synthese_categories_reporting.csv", encoding="utf-8")
print(f"✓ synthese_categories_reporting.csv ({len(reporting)} lignes)")

## 21. Recommandations produit — synthèse actionnable

Traduction des résultats statistiques en actions concrètes pour l'équipe produit Abritel.
Tous les chiffres ci-dessous sont **calculés directement** depuis les données.

In [ ]:
print("=" * 70)
print("SYNTHÈSE ACTIONNABLE — Avis Abritel")
print("=" * 70)

# --- Référence Google Play ---
gp_df = df[df["source"] == "Google Play"]
n_gp = len(gp_df)
prom_gp = 100 * gp_df["note"].isin([4, 5]).sum() / n_gp
detr_gp = 100 * gp_df["note"].isin([1, 2]).sum() / n_gp
nps_gp = round(prom_gp - detr_gp, 1)
print(f"\nRéférence Google Play (n={n_gp}) : NPS-like = {nps_gp:+.1f}")

# --- Priorisation toutes sources ---
prio_final = (
    df.groupby("Catégorie")
    .agg(
        volume=("note", "size"),
        note_moy=("note", "mean"),
        pct_haute=("Gravité", lambda s: 100 * (s == "Haute").sum() / len(s)),
    )
    .round(2)
)
prio_final["score"] = (prio_final["volume"] * prio_final["pct_haute"] / 100).round(1)
prio_final = prio_final.sort_values("score", ascending=False)

print(f"\n{'PRIORITÉS PAR CATÉGORIE':^70}")
print(f"{'-' * 70}")
for cat, row in prio_final.iterrows():
    level = (
        "CRITIQUE" if row["pct_haute"] >= 80 else ("ÉLEVÉ" if row["pct_haute"] >= 50 else "MODÉRÉ")
    )
    print(
        f"  [{level:8s}] {cat:30s} | vol={row['volume']:3.0f} | note={row['note_moy']:.2f} | "
        f"haute={row['pct_haute']:.0f}% | score={row['score']:.0f}"
    )

# --- Angle mort ---
n_autre_final = int((df["Catégorie"] == "Autre").sum())
n_pos_court = int((df["Autre_type"] == "positif court").sum())
n_neutre = int((df["Autre_type"] == "neutre").sum())
n_neg_nc = int((df["Autre_type"] == "négatif non catégorisé").sum())
print(f"\nAngle mort « Autre » : {n_autre_final} avis ({100 * n_autre_final / N:.1f}%)")
print(f"  positif court          : {n_pos_court}")
print(f"  neutre                 : {n_neutre}")
print(f"  négatif non catégorisé : {n_neg_nc} (angle mort réel)")

# --- Apport IA ---
n_corrections_final = int((df["Catégorie_mots_cles"] != df["Catégorie_ollama"]).sum())
print(
    f"\nApport IA (Ollama) : {n_corrections_final} avis reclassés ({100 * n_corrections_final / N:.1f}%)"
)
print(f"Cohen κ = {kappa:.3f}")

# --- Catégories multi-thèmes les plus fréquentes ---
sec_top = (
    df[df["Catégorie_secondaire"].str.strip() != ""]
    .groupby(["Catégorie", "Catégorie_secondaire"])
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)
if not sec_top.empty:
    top_pair = sec_top.iloc[0]
    print(
        f"\nPaire multi-thèmes la plus fréquente : "
        f"{top_pair['Catégorie']} + {top_pair['Catégorie_secondaire']} ({top_pair['n']} avis)"
    )

# --- Tendance ---
print(f"\nTendance Kendall τ = {tau:+.3f} (p = {p_tau:.4f})")
if p_tau < ALPHA:
    print(f"  → Tendance significative à la {'hausse' if tau > 0 else 'baisse'}")
else:
    print("  → Pas de tendance significative sur la période")

print(f"\n{'=' * 70}")
print("ACTIONS RECOMMANDÉES")
print(f"{'=' * 70}")
actions_txt = []
for cat, row in prio_final.iterrows():
    if cat == "Autre":
        continue
    if row["pct_haute"] >= 80:
        actions_txt.append(
            f"  [CRITIQUE] {cat} — {row['volume']:.0f} avis, "
            f"{row['pct_haute']:.0f}% gravité haute, note moy. {row['note_moy']:.2f}"
        )
    elif row["pct_haute"] >= 50:
        actions_txt.append(
            f"  [ÉLEVÉ]    {cat} — {row['volume']:.0f} avis, "
            f"{row['pct_haute']:.0f}% gravité haute, note moy. {row['note_moy']:.2f}"
        )

for a in actions_txt:
    print(a)
if not actions_txt:
    print("  Aucune catégorie avec ≥ 50% de gravité haute.")

## 22. Limites méthodologiques

1. **Biais de sélection par plateforme** (majeur) : Trustpilot et App Store attirent quasi-exclusivement les insatisfaits. Toute statistique agrégée est contaminée par le poids relatif de ces sources. Interpréter les métriques **par source** ou sur Google Play uniquement pour un signal équilibré.

2. **Angle mort positif** *(atténué)* : les mots-clés de catégorisation détectent les *problèmes*, pas les compliments. Les avis positifs courts tombent dans « Autre ». Le sous-typage `Autre_type` distingue désormais **positif court**, **positif thématique** (détecté par mots-clés positifs : « génial », « parfait », « recommande »...), **négatif non catégorisé** et **neutre**.

3. **Double catégorisation** : la catégorie finale est celle d'Ollama (accord ~70% avec les mots-clés, κ de Cohen modéré). L'IA corrige ~30% des cas, principalement des erreurs de mots-clés sur des avis ambigus.

4. **Tautologie Gravité** *(atténuée)* : la `Gravité` standard est calculée depuis la note et la catégorie. La colonne `Gravité_texte` offre un **signal indépendant** basé uniquement sur le vocabulaire du texte (colère, urgence légale, vocabulaire négatif). La section 15b compare les deux et mesure la part tautologique du Cramér V.

5. **Validité du χ²** *(corrigée)* : quand des cellules ont un effectif attendu < 5, un **test exact de Fisher** est appliqué en complément du χ² (section 15). Les p-values Fisher sont plus fiables pour les combinaisons rares.

6. **Causalité** : toutes les corrélations (Spearman, Kendall τ, résidus ASR) sont **associatives**. Aucune ne prouve un lien causal entre une catégorie de plainte et une cause produit.

7. **Volume** : ~700 avis sur ~16 mois ≈ 44 avis/mois. Les sous-groupes (App Store, certaines catégories) sont trop petits pour des conclusions robustes — surveiller leur évolution avant toute décision structurelle.

8. **Analyse textuelle** *(améliorée)* : les tokens sont désormais **lemmatisés** avec `simplemma` (français). Les variantes morphologiques (« problème » / « problèmes ») sont regroupées. Le lift n'est pas testé statistiquement.

9. **Version release** *(améliorée)* : l'attribution version → avis inclut un **délai de grâce de 2 jours**. Un avis publié dans les 48h suivant une release est attribué à la version précédente (l'utilisateur n'a probablement pas encore mis à jour).

---
**Reproductibilité** : tous les indicateurs sont calculés depuis `data/avis_enrichis.csv` avec les mêmes définitions que Power BI. Pour régénérer : `uv run python 1_pipeline.py`